# 재시도·폴백과 감사 로그

일시 오류만 제한적으로 재시도하고, 실패하면 안전한 답으로 전환한 경로를 기록합니다.

In [1]:
%pip install -q langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## 실행 환경 준비

감사 로그에는 합성 질문의 마스킹본과 해시만 남깁니다. 예상 API 비용은 0.01달러 미만입니다.

In [2]:
from pathlib import Path
import os, sys
from dotenv import find_dotenv, load_dotenv

start = Path.cwd().resolve()
candidates = (start, *start.parents)
ROOT = next(path for path in candidates if (path / "labs/part4/ch16-guardrails").is_dir())
LAB = ROOT / "labs/part4/ch16-guardrails"
load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "프로젝트 루트 .env에 OPENAI_API_KEY가 필요합니다."
print("Python:", sys.version.split()[0], "| API 키 준비: True")

Python: 3.12.13 | API 키 준비: True


In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "openai:gpt-5.4-mini", temperature=0, max_tokens=140,
    timeout=30, max_retries=0,
)
print("모델 준비:", type(model).__name__)

모델 준비: ChatOpenAI


## 일시 오류 재현

첫 두 호출이 시간 초과라고 가정합니다.

In [4]:
from typing import Callable

def retry_with_fallback(primary: Callable[[], str], fallback: Callable[[], str], attempts: int = 2):
    for index in range(1, attempts + 1):
        try:
            return primary(), "primary", index
        except TimeoutError:
            pass
    return fallback(), "fallback", attempts + 1

state = {"calls": 0}
def flaky():
    state["calls"] += 1
    if state["calls"] < 3:
        raise TimeoutError("합성 시간 초과")
    return "기본 모델 성공"

result = retry_with_fallback(flaky, lambda: "폴백 답", attempts=2)
print("결과·경로·시도:", result)

결과·경로·시도: ('폴백 답', 'fallback', 3)


## 실제 모델 폴백 경로

기본 함수가 영구 오류를 낸 상황에서 실제 모델을 폴백 함수로 사용합니다.

In [5]:
def permanent_failure():
    raise TimeoutError("합성 제공자 장애")

def fallback_model():
    return model.invoke("서비스 지연 안내를 한 문장으로 작성하세요.").text

text, route, attempts = retry_with_fallback(
    permanent_failure, fallback_model, attempts=1,
)
print(route, attempts, text)

fallback 2 현재 서비스 지연이 발생하고 있으며, 신속히 정상화될 수 있도록 최선을 다하고 있습니다.


## 감사 로그

원문 PII를 저장하지 않고 정책 결과와 경로를 기록합니다.

In [6]:
import hashlib, json, re

def safe_audit(query: str, event: str, detail: dict[str, str]) -> dict[str, object]:
    masked = re.sub(r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}", "<EMAIL_MASKED>", query)
    safe_detail = {key: re.sub(r"01\d[ -]?\d{3,4}[ -]?\d{4}", "<PHONE_MASKED>", value)
                   for key, value in detail.items()}
    return {"event": event, "query_masked": masked,
            "query_hash": hashlib.sha256(("ch16-session" + masked).encode()).hexdigest()[:12],
            "detail": safe_detail}

audit_path = LAB / "runs/05-audit.jsonl"
audit_path.unlink(missing_ok=True)
audit_path.parent.mkdir(exist_ok=True)
row = safe_audit("test@example.com 문의", "fallback", {"route": route})
audit_path.write_text(json.dumps(row, ensure_ascii=False) + "\n")
print(row)
print("원문 이메일 저장:", "test@example.com" in audit_path.read_text())

{'event': 'fallback', 'query_masked': '<EMAIL_MASKED> 문의', 'query_hash': 'fb7571919033', 'detail': {'route': 'fallback'}}
원문 이메일 저장: False


## 반복 실험과 결과 해석

일시 오류 뒤 고정 답 폴백과 영구 오류 뒤 실제 모델 폴백을 서로 다른 사건으로 기록해야 원인을 찾을 수 있습니다.

In [7]:
print("로컬 재시도 호출:", state["calls"])
print("폴백 경로:", route)
print("감사 사건 수:", len(audit_path.read_text().splitlines()))
assert route == "fallback" and "test@example.com" not in audit_path.read_text()

로컬 재시도 호출: 2
폴백 경로: fallback
감사 사건 수: 1
